# Reproduce `submission_BC.csv` in Google Colab

This notebook regenerates the **exact** winning prediction file `submission_BC.csv` (evaluation score **91.25092**, where score = `100 * (1 - RMSE)`) by running the self-contained, deterministic script `reproduce.py`.

`reproduce.py` reads `train.csv` + `test.csv`, builds model **B** (feature-enhanced LightGBM, seeds 42-47, 437 rounds) and model **C** (`0.406*LightGBM[217 rounds] + 0.594*XGBoost[geohash dropped, 259 rounds]`, seeds 42/143/244/345), averages them 50/50, clips to `[0, 1]`, and writes `submission_BC.csv` (**41778 rows**, columns `Index,demand`). All seeds are fixed, so the output is reproducible.

**Runtime:** a CPU runtime is fine (no GPU needed); the run takes a couple of minutes.

## Expected file layout

`reproduce.py` locates the data by probing several directories, including `./dataset` (next to the script) and `/content/dataset`. It writes `submission_BC.csv` **next to `reproduce.py`**. Put `reproduce.py` in `/content/` and the CSVs in `/content/dataset/`:

```
/content/
  reproduce.py
  dataset/
    train.csv
    test.csv
```

Output will be `/content/submission_BC.csv`.

## Step 1 - Upload the files

Run the cell below and select all three files in the dialog: `reproduce.py`, `train.csv`, `test.csv`. (Alternatively, drag-and-drop them into the **Files** pane on the left.)

In [ ]:
from google.colab import files
uploaded = files.upload()   # select reproduce.py, train.csv, test.csv

## Step 2 - Arrange the dataset folder

Move the two CSVs into `/content/dataset/` so the layout matches what the script expects, then confirm.

In [ ]:
!mkdir -p /content/dataset
!mv /content/train.csv /content/test.csv /content/dataset/ 2>/dev/null || true
!ls -la /content /content/dataset

### Alternative: load from Google Drive

If your files already live in Drive, use this instead of the upload steps above (edit `SRC` to your folder). Otherwise skip this cell.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# SRC = '/content/drive/MyDrive/gridlock'   # <-- edit to your folder
# !mkdir -p /content/dataset
# !cp "{SRC}/reproduce.py" /content/reproduce.py
# !cp "{SRC}/train.csv" /content/dataset/train.csv
# !cp "{SRC}/test.csv"  /content/dataset/test.csv
# !ls -la /content /content/dataset

## Step 3 - (Optional) Install dependencies

`reproduce.py` auto-installs anything missing, so this is usually not needed. Run it only as a fallback.

In [ ]:
!pip install numpy pandas scikit-learn lightgbm xgboost

## Step 4 - Run the reproducer

This takes a couple of minutes on a CPU runtime. The line `verify: no reference file found -- skipping comparison` is normal in Colab. Watch for the final `wrote /content/submission_BC.csv rows=41778`.

In [ ]:
!python /content/reproduce.py

## Step 5 - Sanity-check the output

Because the seeds are fixed, this file matches the **91.25092** submission. Confirm the invariants: 41778 rows, columns `Index,demand`, demand within `[0, 1]`, no NaNs.

In [ ]:
import pandas as pd
sub = pd.read_csv('/content/submission_BC.csv')
print('rows:', len(sub))                 # expect 41778
print('columns:', list(sub.columns))     # expect ['Index', 'demand']
print('demand min/max:', sub['demand'].min(), sub['demand'].max())  # within [0, 1]
print('any NaN:', bool(sub.isna().any().any()))                     # expect False
sub.head()

## Step 6 - Download `submission_BC.csv`

In [ ]:
from google.colab import files
files.download('/content/submission_BC.csv')